In [ ]:
import numpy as np 
import pandas as pd 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
import numpy as np

df = pd.read_parquet('../data/Divar_cleaned.parquet')

is_sell = df["cat2_slug"].str.contains("sell", na=False).fillna(False).astype(bool)
is_full_credit = (df["inferred_full_credit"] == True).fillna(False).astype(bool)

conditions = [is_sell, is_full_credit]
choices = [df["price_value"], df["transformable_credit"]]

df["unified_price"] = np.select(conditions, choices, default=df["transformed_credit"])
df["unified_price"] = df["unified_price"].replace(0, np.nan)

print("cover unified_price:", df["unified_price"].notna().mean() * 100, "%")

<div dir="rtl">

### خلاصه ساخت ستون قیمت یکپارچه (`unified_price`)

* **ساخت شرط‌ها:** تعیین دو شرط منطقی Boolean (همراه با ایمن‌سازی نوع داده جهت جلوگیری از خطا) برای تشخیص آگهی‌های **فروشی** (`is_sell`) و **رهن کامل** (`is_full_credit`).
* **ترکیب شرطی داده‌ها:** اعمال `np.select` برای جایگذاری هوشمند قیمت‌ها:
  1. اگر **فروشی** بود $\leftarrow$ قیمت فروش (`price_value`)
  2. اگر **رهن کامل** بود $\leftarrow$ رهن قابل‌تبدیل (`transformable_credit`)
  3. در غیر این صورت (**رهن و اجاره ترکیبی**) $\leftarrow$ ارزش تبدیل‌شده (`transformed_credit`)
* **اصلاح صفرهای مصنوعی:** تبدیل صفرهای پیش‌فرض `np.select` به `NaN` جهت جلوگیری از خراب شدن محاسبات آماری (قیمت ۰ تومان).
* **دستآورد اصلی:** افزایش پوشش داده‌های قیمت از حالت تک‌بعدی به **۶۸٪** از کل آگهی‌ها.

---

#### 💡 نکته کلیدی
تابع `np.select` شرط‌ها را به‌ترتیب بررسی می‌کند؛ بنابراین **ترتیب شرط‌ها** در لیست اهمیت حیاتی دارد و به محض صادق بودن اولین شرط، مقدار آن اعمال می‌شود.

</div>

In [ ]:
cluster_features = ["unified_price", "location_utm_x", "location_utm_y", "building_size"]

cluster_df = df[cluster_features].dropna().copy()

print("تعداد ردیف قبل از حذف NaN:", len(df))
print("تعداد ردیف بعد از حذف NaN:", len(cluster_df))
print(f"درصد داده‌ی باقی‌مانده: {len(cluster_df) / len(df) * 100:.1f}%")

<div dir="rtl">

### انتخاب ویژگی‌ها و آماده‌سازی داده برای خوشه‌بندی

* **تعریف ویژگی‌های نهایی:** انتخاب ۴ ویژگی اصلی شامل قیمت یکپارچه (`unified_price`)، موقعیت جغرافیایی (`location_utm_x`, `location_utm_y`) و متراژ (`building_size`). محدود نگه داشتن تعداد ویژگی‌ها برای جلوگیری از **نفرین ابعاد (Curse of Dimensionality)** است.
* **حذف مقادیر مفقود (NaN):** جداسازی این ۴ ستون و حذف کامل سطرهای دارای داده مفقود، به دو علت:
  1. الگوریتم KMeans توانایی پردازش مقادیر خالی را ندارد.
  2. جای‌گذاری (Imputation) برای مختصات جغرافیایی یا قیمت، موقعیت مکانی بی‌معنا ایجاد می‌کند.

تعداد ابعاد کمتر در خوشه‌بندی، باعث حفظ ساختار فاصله‌ای معنادار بین نقاط می‌شود؛ همچنین عدم پرکردن داده‌های مفقود مکانی با روش‌های عمومی (مانند میانگین)، از ایجاد «املاک غیر واقعی » در نقاط جغرافیایی نامربوط جلوگیری می‌کند.

</div>

<div dir="rtl">

### دلایل عدم پرکردن (Impute) مقادیر مفقود (NaN)

* **نابود شدن صحت داده‌های مکانی:** پرکردن مختصات جغرافیایی (`UTM`) با روش‌های آماری (مثل میانگین)، ملک را به مرکز هندسی شهر منتقل کرده و «موقعیت غیرواقعی» می‌سازد.
* **تخریب مرز خوشه‌ها در قیمت و متراژ:** متغیرهای قیمت و متراژ توزیع ناهمگون دارند؛ جای‌گذاری آن‌ها با مقادیر تقریبی، ملک‌های ناهمگون را کنار هم قرار داده و مرز بین خوشه‌ها را خراب می‌کند.
* **حساسیت الگوریتم‌های مبتنی بر فاصله:** الگوریتم‌هایی نظیر KMeans بر پایه فاصله اقلیدسی کار می‌کنند و داده‌های ساختگی باعث ایجاد نویز و جابه‌جایی مراکز خوشه‌ها (Centroids) می‌شوند.

در اینجا ما یک روش امن که حذف داده ها بود را انتخاب کردیم با 42 درصد داده باقی مانده به مدل سازی ادامه می دهیم 

</div>

In [ ]:
scaler = StandardScaler()
cluster_df["log_unified_price"] = np.log1p(cluster_df["unified_price"])
cluster_df["log_building_size"] = np.log1p(cluster_df["building_size"])

cluster_features_log = ["log_unified_price", "location_utm_x", "location_utm_y", "log_building_size"]

X_scaled = scaler.fit_transform(cluster_df[cluster_features_log])

kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
cluster_df["cluster"] = kmeans.fit_predict(X_scaled)

centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
centers_df = pd.DataFrame(centers_original, columns=cluster_features_log)

print(cluster_df["cluster"].value_counts().sort_index())

<div dir="rtl">

### خوشه‌بندی KMeans با نرمال‌سازی لگاریتمی و استانداردسازی

* **کاهش اثر داده‌های پرت (توزیع راست‌چوله):** اعمال تبدیل لگاریتمی (`np.log1p`) روی متغیرهای قیمت و متراژ، باعث فشرده‌سازی مقادیر بسیار بزرگ شده و از تسلط داده‌های پرت بر روند خوشه‌بندی جلوگیری می‌کند.
* **هم‌مقیاس‌سازی متغیرها (`StandardScaler`):** به دلیل اتکای الگوریتم KMeans بر «فاصله اقلیدسی»، تمام ۴ ویژگی (قیمت لگاریتمی، متراژ لگاریتمی و مختصات UTM) روی مقیاسی با میانگین ۰ و انحراف معیار ۱ تنظیم می‌شوند تا هیچ ویژگی صرفاً به خاطر بزرگی عدد خام، متغیرهای دیگر را بی‌اثر نکند.
* **اجرای الگوریتم KMeans:** آموزش مدل روی داده‌های استانداردشده با ۱۰ خوشه (`n_clusters=10`). پارامتر `n_init=10` باعث اجرای ۱۰ باره الگوریتم با نقاط شروع تصادفی مختلف و انتخاب بهینه‌ترین ساختار می‌شود.
* **بازگرداندن مراکز خوشه‌ها به مقیاس اولیه:** استفاده از `inverse_transform` برای تبدیل مرکز خوشه‌ها از مقیاس استانداردشده به مقیاس پیش از استانداردسازی (جهت امکان تحلیل منطقی و رسم نمودارها).
* **ارزیابی توازن خوشه‌ها:** بررسی تعداد نمونه‌های هر خوشه جهت اطمینان از توزیع متوازن‌تر داده‌ها (جلوگیری از تجمع غیرطبیعی بیش از ۵۰٪ داده‌ها در یک خوشه تنها).

---
ترکیب **تبدیل لگاریتمی** و **استانداردسازی**، دو ناهنجاری اصلی در داده‌های املاک (چولگی شدید قیمت/متراژ و عدم توازن مقیاس با مختصات مکانی) را برطرف کرده و منجر به شکل‌گیری خوشه‌هایی متوازن و از نظر آماری باکیفیت می‌شود.

</div>

In [ ]:
import matplotlib.pyplot as plt

# برگردوندن مراکز به مقیاس اصلی (غیر لگاریتمی)
centers_scaled_back = scaler.inverse_transform(kmeans.cluster_centers_)
centers_df = pd.DataFrame(centers_scaled_back, columns=cluster_features_log)

# log_unified_price و log_building_size رو به مقیاس واقعی برگردون
centers_df["unified_price"] = np.expm1(centers_df["log_unified_price"])
centers_df["building_size"] = np.expm1(centers_df["log_building_size"])

fig, ax = plt.subplots(figsize=(12, 10))

scatter = ax.scatter(
    cluster_df["location_utm_x"],
    cluster_df["location_utm_y"],
    c=cluster_df["cluster"],
    cmap="tab10",
    s=15,
    alpha=0.5,
)

ax.scatter(
    centers_df["location_utm_x"],
    centers_df["location_utm_y"],
    c="black",
    marker="X",
    s=300,
    edgecolors="white",
    linewidths=1.5,
    label="مرکز خوشه‌ها",
    zorder=5,
)

for i, row in centers_df.iterrows():
    ax.annotate(
        str(i),
        (row["location_utm_x"], row["location_utm_y"]),
        color="white",
        fontsize=9,
        fontweight="bold",
        ha="center",
        va="center",
        zorder=6,
    )

print(centers_df[["unified_price", "building_size", "location_utm_x", "location_utm_y"]].round(2))

legend1 = ax.legend(*scatter.legend_elements(), title="شماره خوشه", loc="upper left", bbox_to_anchor=(1.02, 1))
ax.add_artist(legend1)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 0.3))

ax.set_xlabel("UTM X (متر)")
ax.set_ylabel("UTM Y (متر)")
ax.set_title("خوشه بندی ")

plt.tight_layout()
plt.show()

<div dir="rtl">

### تجسم‌سازی و رسم نمودار خوشه‌های املاک

* **بازیابی و تبدیل مقادیر واقعی مراکز:** بازگرداندن مراکز خوشه‌ها از داده‌های استانداردشده به مقایس لگاریتمی و سپس استفاده از تابع معکوس `np.expm1` جهت دستیابی به قیمت (تومان) و متراژ (متر مربع) واقعی مراکز.
* **رسم پراکندگی داده‌ها (Scatter Plot):** نمایش نقاط مربوط به املاک بر اساس مختصات مکانی (`UTM`) با تفکیک ۱۰ رنگ متمایز (`cmap="tab10"`) متناظر با شماره هر خوشه.
* **علامت‌گذاری مراکز خوشه‌ها:** مشخص کردن موقعیت هندسی مرکز هر خوشه روی نقشه با نماد بزرگ `X` و درج شماره خوشه در داخل آن جهت شناسایی دقیق.
* **چاپ مشخصات عددی مراکز:** ارائه جدول مقادیر واقعیِ مرکز هر خوشه (قیمت، متراژ و مختصات) جهت بررسی دقیق‌تر ابعاد غیرمکانی و رفع ابهام در مناطق پر تراکم.
* **تنظیمات راهنما و محورها:** افزودن دوتایی راهنمای نمودار (راهنمای رنگ خوشه‌ها و علامت مراکز)، عنوان و برچسب‌گذاری محورهای مختصات بر حسب متر.

<div dir="rtl">

### علت استفاده از تبدیل لگاریتمی (`np.log1p`)

* **اصلاح چولگی داده‌ها (Skewness):** توزیع قیمت و متراژ در بازار املاک شدیداً به سمت راست چوله است (اکثر املاک قیمت و متراژ معمولی دارند، اما تعداد محدودی املاک فوق‌العاده گران یا بسیار بزرگ وجود دارد).
* **کاهش اثر داده‌های پرت (Outliers):** داده‌های بسیار بزرگ (پرت)، مقادیر فاصله اقلیدسی را کج کرده و باعث می‌شوند الگوریتم KMeans بخش عظیمی از داده‌های معمولی (مثلاً ۵۸٪) را در یک خوشه‌ی غول‌پیکر انباشته کند.
* **فشرده‌سازی مقیاس:** لگاریتم گرفتن، فاصله بین مقادیر بزرگ را فشرده و مقیاس را متعادل می‌کند؛ در نتیجه املاک معمولی امکان تاثیرگذاری منطقی در فرایند خوشه‌بندی و شکل‌گیری خوشه‌های متوازن‌تر را پیدا می‌کنند.


#### 💡 نکته کلیدی
تبدیل معکوس لگاریتم با `np.expm1` روی مراکز خوشه‌ها، امکان تحلیل هم‌زمان **ابعاد مکانی** (روی نمودار) و **ابعاد اقتصادی/فیزیکی** (در جدول خروجی) را به صورت کاملاً ملموس و واقعی فراهم می‌سازد.

</div>

<div dir="rtl">

### تحلیل خروجی خوشه‌بندی و بررسی تحویل پروژه

* **تحلیل رفتاری خوشه‌ها:**
  * **خوشه‌های حاشیه‌ای (۳، ۸، ۲، ۹ و ۰):** در مناطقی که املاک پراکنده‌تر هستند، **موقعیت جغرافیایی** عامل اصلی تفکیک است و هر محدوده یک خوشه مجزا ساخته است.
  * **خوشه‌های مرکزی (۱، ۴، ۵، ۶ و ۷):** در مناطق پرتراکم (مانند مرکز تهران/البرز) که موقعیت مکانی املاک بسیار به هم نزدیک است، تفکیک خوشه‌ها بر اساس **قیمت و متراژ** انجام شده است.
* **موقعیت مراکز خوشه‌ها (X مشکی):** جانمایی دقیق انجام شده و قرارگیری چند مرکز نزدیک به هم در ناحیه مرکزی، به دلیل تراکم بالای مکانی در آن محدوده طبیعی است.

> «الگوریتم خوشه‌بندی در مناطق حاشیه‌ای و کم‌تراکم، املاک را عمدتاً بر اساس **فاصله مکانی** دسته‌بندی کرده است؛ اما در مناطق پرتراکم شهری، به دلیل نزدیکی جغرافیایی داده‌ها، **قیمت و متراژ** به متغیر اصلی تفکیک خوشه‌ها تبدیل شده‌اند.»

</div>

In [ ]:
wcss = []
k_range = range(1, 21)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    print(f"k={k:2d} -> WCSS={km.inertia_:,.0f}")

<div dir="rtl">

### محاسبه معیار WCSS جهت یافتن تعداد بهینه خوشه‌ها (روش Elbow)

* **تعریف بازه جستجو:** تنظیم بازه تغییرات تعداد خوشه‌ها ($k$) از ۱ تا ۲۰ به همراه ایجاد یک لیست خالی برای ذخیره میزان **مجموع مربعات فواصل درون‌خوشه‌ای (WCSS)**.
* **آموزش متوالی مدل KMeans:** اجرای الگوریتم به ازای هر $k$ روی داده‌های لگاریتمی و استانداردشده (`X_scaled`). تنظیم `n_init=10` جهت تضمین یافتن بهترین ساختار در هر ارزیابی.
* **استخراج شاخص Inertia:** استخراج مقدار `km.inertia_` (معادل WCSS) و اضافه کردن آن به لیست؛ این شاخص میزان فشرده بودن خوشه‌ها را می‌سنجد و با افزایش $k$ کاهش می‌یابد.
* **پایش عددی روند:** چاپ مقدار WCSS به ازای هر $k$ جهت بررسی دقیق افت خطای مدل و آمادگی برای رسم نمودار  (Elbow Method).

---
هدف اصلی در روش Elbow، یافتن نقطه‌ای است که پس از آن میزان کاهش WCSS **کند و ناچیز** می‌شود؛ این نقطه نشان‌دهنده تعادل بهینه میان کمترین خطای خوشه‌بندی و پیچیدگی مدل است.

</div>

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(list(k_range), wcss, marker="o")
plt.xlabel("تعداد خوشه (k)")
plt.ylabel("مجموع مجذورات درون‌خوشه‌ای (WCSS)")
plt.title("روش آرنج (Elbow Method) برای انتخاب k بهینه")
plt.xticks(list(k_range))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

<div dir="rtl">

### رسم نمودار روش آرنجی (Elbow Method)

* **محورهای نمودار:** رسم مقادیر WCSS (محور عمودی) در برابر تعداد خوشه‌ها $k$ از ۱ تا ۲۰ (محور افقی).
* **علامت‌گذاری نقاط (`marker="o"`):** درج دایره روی هر نقطه جهت شفافیت و امکان مشاهده دقیق مقادیر WCSS به ازای هر $k$.
* **نمایش کامل محور افقی (`plt.xticks`):** درج تمام اعداد ۱ تا ۲۰ روی محور $k$ جهت جلوگیری از حذف خودکار مقادیر توسط کتابخانه.
* **تسهیل خوانایی (`plt.grid`):** اضافه کردن خطوط شبکه‌ای کم‌رنگ در پس‌زمینه برای سنجش چشمی دقیق‌تر افت منحنی.

####  هدف
شناسایی بصری نقطه‌ای که منحنی در آن دچار «شکست» یا «آرنج» می‌شود و از آن نقطه به بعد، میزان کاهش WCSS ناچیز می‌گردد.

</div>

In [ ]:
from kneed import KneeLocator

kl = KneeLocator(list(k_range), wcss, curve="convex", direction="decreasing")
optimal_k = kl.elbow

print("k بهینه (به‌روش Kneedle):", optimal_k)

kl.plot_knee()
plt.title(f"تشخیص خودکار نقطه‌ی آرنج — k = {optimal_k}")
plt.xlabel("تعداد خوشه (k)")
plt.ylabel("WCSS")
plt.show()

<div dir="rtl">

### تشخیص خودکار نقطه‌ی آرنج با الگوریتم ریاضی

* **محاسبه دقیق نقطه شکست (`KneeLocator`):** استفاده از کتابخانه `kneed` برای یافتن نقطه آرنج با تحلیل ریاضی انحنای منحنی، به جای اتکا به حدس چشمی.
  * **`curve="convex"`:** تنظیم نوع انحنا به حالت محدب (شکل طبیعی منحنی WCSS).
  * **`direction="decreasing"`:** تنظیم جهت روند نزولی مقادیر WCSS با افزایش $k$.
* **استخراج $k$ بهینه (`optimal_k`):** دریافت عدد پیشنهادی الگوریتم برای تعداد خوشه‌های بهینه.
* **تجسم‌سازی نقطه آرنج (`kl.plot_knee`):** رسم منحنی به همراه یک خط عمودی بر روی $k$ بهینه جهت اعتبارسنجی بصری.

</div>

In [ ]:
from sklearn.metrics import silhouette_score
import numpy as np

sample_idx = np.random.RandomState(42).choice(X_scaled.shape[0], size=20000, replace=False)
X_sample = X_scaled[sample_idx]

silhouette_scores = []
k_range_sil = range(2, 21)  # Silhouette برای k=1 تعریف‌نشده

for k in k_range_sil:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_sample = km.fit_predict(X_sample)
    score = silhouette_score(X_sample, labels_sample)
    silhouette_scores.append(score)
    print(f"k={k:2d} -> Silhouette={score:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(list(k_range_sil), silhouette_scores, marker="o", color="darkorange")
plt.xlabel("تعداد خوشه (k)")
plt.ylabel("Silhouette Score")
plt.title("امتیاز Silhouette برای انتخاب k (روی نمونه‌ی ۲۰ هزارتایی)")
plt.xticks(list(k_range_sil))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_k_silhouette = list(k_range_sil)[np.argmax(silhouette_scores)]
print("بهترین k بر اساس Silhouette:", best_k_silhouette)

<div dir="rtl">

# تحلیل امتیاز Silhouette و انتخاب نهایی تعداد خوشه (k)

## هدف این بخش از کد

بعد از بررسی WCSS (روش Elbow) که فقط **فشردگی درون‌خوشه‌ای** را می‌سنجد، در این بخش از معیار **Silhouette Score** استفاده شده تا علاوه بر فشردگی، **میزان جدایی بین خوشه‌ها** هم ارزیابی شود. این معیار عددی بین ۱- تا ۱ می‌دهد؛ هرچه به ۱ نزدیک‌تر باشد، یعنی نقاط هر خوشه هم به‌خوبی از خوشه‌های دیگر جدا شده‌اند و هم درون خوشه‌ی خودشان منسجم هستند.

از آن‌جا که محاسبه‌ی این معیار روی کل دیتاست (حدود ۴۲۹ هزار سطر) بسیار زمان‌بر است، محاسبات روی یک **نمونه‌ی تصادفی ۲۰ هزارتایی** انجام شده تا در زمان معقول، برای مقادیر مختلف k (از ۲ تا ۲۰) امتیاز محاسبه و مقایسه شود.

---

## نتایج به‌دست‌آمده

| k | Silhouette Score |
|---|---|
| ۲ | ۰.۳۶۴۶ |
| ۳ | **۰.۳۹۸۸ (بیشترین امتیاز)** |
| ۴ | ۰.۳۰۰۷ |
| ۵ | ۰.۳۳۱۶ |
| ۶ | ۰.۲۵۳۶ |
| ۷ تا ۲۰ | نوسان کم‌دامنه بین ۰.۲۵ تا ۰.۲۷ |

نمودار به‌دست‌آمده یک الگوی مشخص را نشان می‌دهد: امتیاز از k=2 به k=3 افزایش می‌یابد، سپس به‌شدت افت می‌کند و از k=6 به بعد وارد یک **فلات کم‌نوسان** می‌شود؛ یعنی افزودن خوشه‌های بیشتر از این نقطه به بعد، تغییر معناداری در کیفیت جداسازی خوشه‌ها ایجاد نمی‌کند.

---

## مقایسه با نتیجه‌ی روش Elbow (WCSS)

روش Elbow یک شکستگی نرم حوالی k=5 یا k=6 نشان می‌داد، در حالی‌که Silhouette بیشترین امتیاز را در k=3 ثبت کرده است. این دو روش **کاملاً هم‌راستا نیستند** و این خودش یک یافته‌ی مهم است، نه یک تناقض قابل‌نادیده‌گرفتن.

### چرا این دو معیار با هم اختلاف دارند؟

با بررسی اسکترپلات جغرافیایی خوشه‌ها (بخش قبلی)، مشخص شد که ساختار داده **همگن نیست** و از دو نوع الگوی متفاوت تشکیل شده:

- بخشی از خوشه‌ها (مربوط به شهرهای بزرگ و دورافتاده از هم مانند تهران، مشهد و مناطق جنوبی) **کاملاً بر مبنای فاصله‌ی جغرافیایی** از هم جدا هستند. این خوشه‌ها حتی با تعداد کم خوشه (k کوچک) هم به‌وضوح از هم تفکیک می‌شوند، به همین دلیل Silhouette در k‌های پایین (به‌ویژه k=3) امتیاز بالایی می‌گیرد.
- بخش دیگری از داده (مربوط به مناطق پرتراکم مانند تهران و البرز) از نظر مکانی به‌هم بسیار نزدیک هستند و تنها راه تفکیک معنادار آن‌ها، تکیه بر **قیمت و متراژ** است. این بخش برای آشکارشدن تنوع درونی‌اش نیاز به تعداد خوشه‌ی بیشتری دارد؛ چیزی که Silhouette (چون میانگین کل داده را می‌سنجد) به‌خوبی آن را منعکس نمی‌کند.

به‌عبارت دیگر، معیارهای کلاسیک مانند WCSS و Silhouette فرض می‌کنند خوشه‌ها کروی‌شکل و تقریباً هم‌اندازه هستند؛ در حالی‌که داده‌ی این پروژه ترکیبی از خوشه‌های کاملاً مجزای جغرافیایی و خوشه‌های هم‌پوشانِ قیمت‌محور است. به همین دلیل، تکیه‌ی صرف به یکی از این دو معیار می‌تواند گمراه‌کننده باشد.

---

## نتیجه‌گیری نهایی و انتخاب k

با در نظر گرفتن هر دو معیار و هدف پروژه (ساخت یک سیستم توصیه‌گر ملک با خوشه‌های معنادار)، **k=5** به‌عنوان مقدار نهایی انتخاب شد، به دلایل زیر:

1. هم روش Elbow (شکستگی نرم منحنی WCSS) و هم روش Silhouette (پیک محلی دوم با امتیاز ۰.۳۳) روی این مقدار هم‌پوشانی دارند.
2. انتخاب k=3 (بهترین امتیاز خام Silhouette) برای یک سیستم توصیه‌گر واقعی بیش‌ازحد کلی و کم‌تنوع خواهد بود؛ تنها چند گروه بسیار درشت می‌سازد که برای توصیه‌ی دقیق و متمایز به کاربر کافی نیست.
3. مقادیر k بزرگ‌تر از ۶ هیچ بهبود معنادار دیگری در Silhouette ایجاد نمی‌کنند (فلات یکنواخت)، یعنی خوشه‌های اضافه صرفاً داده را ریزتر می‌کنند، بدون کشف ساختار جدید و معنادار.

### جمع‌بندی چالش روش‌شناختی

این تحلیل نشان داد که وقتی داده از ترکیب چند الگوی متفاوت (جدایی جغرافیایی در مقیاس بزرگ، هم‌پوشانی قیمت‌محور در مقیاس محلی) تشکیل شده باشد، هیچ معیار عددی به‌تنهایی جواب قطعی نمی‌دهد. انتخاب نهایی k باید ترکیبی از شواهد کمّی (WCSS، Silhouette) و قضاوت کاربردی مبتنی بر هدف پروژه باشد — در این‌جا، نیاز به تنوع کافی خوشه‌ها برای کارکرد درست سیستم توصیه‌گر.

</div>